# Crop Disease Model Test

Use the **Python (CropCare GPU)** kernel. It points to the project Python 3.10 environment with TensorFlow DirectML. A normal Python 3.12 kernel will not see that TensorFlow installation.

This notebook loads the final 17-output ResNet50 model, predicts one uploaded or path-selected image, and applies the same invalid-image gate as the Android app.

## Check the notebook kernel

This cell displays the Python environment used by this notebook. TensorFlow must be installed into this same environment.


In [ ]:
import sys
import importlib.util
from pathlib import Path

print("Python kernel:", sys.executable)
print("Notebook working folder:", Path.cwd())
print("TensorFlow installed in this kernel:", importlib.util.find_spec("tensorflow") is not None)
if importlib.util.find_spec("tensorflow") is None:
    print("Select the Python (CropCare GPU) kernel. It uses the project .tf_directml_venv environment.")


## Load the saved model


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
from PIL import Image, ImageOps

try:
    import tensorflow as tf
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"TensorFlow is not installed in this kernel ({sys.executable}). "
        "Select the Python (CropCare GPU) kernel; it uses the project .tf_directml_venv environment."
    ) from exc

from IPython.display import display

SEARCH_ROOTS = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_DIR = next((p for p in SEARCH_ROOTS if (p / "model_output" / "retrained_balanced_2000_gpu" / "resnet50_leafgate.keras").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find model_output/retrained_balanced_2000_gpu/resnet50_leafgate.keras. Open this notebook from the project folder.")

OUTPUT_DIR = PROJECT_DIR / "model_output" / "retrained_balanced_2000_gpu"
MODEL_PATH = OUTPUT_DIR / "resnet50_leafgate.keras"
CLASS_NAMES_PATH = OUTPUT_DIR / "class_names.json"
METADATA_PATH = OUTPUT_DIR / "model_metadata.json"
IMAGE_SIZE = (224, 224)

class_names = json.loads(CLASS_NAMES_PATH.read_text(encoding="utf-8"))
model_metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
gate = model_metadata["invalid_image_gate"]
NOT_LEAF_LABEL = "__not_a_supported_leaf__"
not_leaf_index = class_names.index(NOT_LEAF_LABEL)
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
if model.output_shape[-1] != len(class_names):
    raise ValueError(f"Model has {model.output_shape[-1]} outputs but there are {len(class_names)} labels.")
print("Loaded model:", MODEL_PATH)
print(f"Recognizes {len(class_names) - 1} leaf classes and one invalid-image class")
print("Expected image size: 224 x 224 RGB")
print("Invalid-image probability threshold:", gate["not_leaf_probability_threshold"])


## Upload a leaf or test image

Upload one clear leaf image or an unrelated image to check the invalid-image gate. You can also set `IMAGE_PATH` to an image file.

In [ ]:
IMAGE_PATH = None
image_uploader = None

try:
    import ipywidgets as widgets
    image_uploader = widgets.FileUpload(accept="image/*", multiple=False)
    display(image_uploader)
    print("Upload one leaf photo using the control above.")
except ImportError:
    IMAGE_PATH = Path(input("Enter the full path to a leaf image: ").strip().strip('"'))


## Predict and apply the invalid-image gate

The same rule as the app is applied: a leaf label must rank above the invalid label, and the invalid-image score must remain below the saved threshold. Scores are not calibrated field-diagnosis probabilities.

In [ ]:
import io

def get_image_bytes():
    if image_uploader is not None:
        uploaded = image_uploader.value
        if uploaded:
            if isinstance(uploaded, dict):
                metadata = next(iter(uploaded.values()))
            else:
                metadata = uploaded[0]
            return bytes(metadata["content"]), metadata.get("name", "uploaded_image")

    path = IMAGE_PATH
    if path is None:
        path = Path(input("No upload found. Enter the full path to an image: ").strip().strip("\""))
    path = Path(path).expanduser()
    if not path.is_absolute():
        path = PROJECT_DIR / path
    if not path.is_file():
        raise FileNotFoundError(f"Image file not found: {path}")
    return path.read_bytes(), path.name

image_bytes, image_name = get_image_bytes()
with Image.open(io.BytesIO(image_bytes)) as source_image:
    original_image = ImageOps.exif_transpose(source_image).convert("RGB")
    resized_image = original_image.resize(IMAGE_SIZE, Image.Resampling.BILINEAR)

pixels = np.asarray(resized_image, dtype=np.float32)
model_input = tf.keras.applications.resnet50.preprocess_input(pixels)
scores = model.predict(model_input[np.newaxis, ...], verbose=0)[0]
if not np.isfinite(scores).all():
    raise ValueError("The model returned a non-finite prediction score.")

leaf_indices = [i for i, label in enumerate(class_names) if label != NOT_LEAF_LABEL]
best_leaf_index = max(leaf_indices, key=lambda i: float(scores[i]))
best_model_index = int(np.argmax(scores))
invalid_probability = float(scores[not_leaf_index])
accepted = (
    best_model_index != not_leaf_index
    and invalid_probability < float(gate["not_leaf_probability_threshold"])
    and float(scores[best_leaf_index]) >= float(gate["minimum_leaf_confidence"])
)

display(original_image)
print("Image:", image_name)
if not accepted:
    print("Invalid image: no supported crop leaf detected. No disease advice is shown.")
    print(f"Not-leaf score: {invalid_probability:.2%}")
else:
    print("Top disease/healthy predictions:")
    for rank, index in enumerate(sorted(leaf_indices, key=lambda i: float(scores[i]), reverse=True)[:3], start=1):
        readable_label = class_names[index].replace("___", " - ").replace("_", " ")
        print(f"{rank}. {readable_label}: {scores[index] * 100:.2f}%")
